In [2]:
import os
import clip
import torch
from torchvision.datasets import CIFAR100
from PIL import Image
import os
import random
from tqdm import tqdm
import sys
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st
import torch
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from k_means_constrained import KMeansConstrained
import seaborn as sns
import seaborn as sns
from collections import Counter
from math import ceil

from utils import (
    Config,
    dataset_object,
    evaluate_predictions,
    get_class_names,
    get_labeled_and_unlabeled_data,
    save_parameters,
    save_predictions,
    store_results,
)

plt.rcParams['font.family'] = 'Nimbus Roman'


In [3]:
# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
print(f"Using device: {device}")
model, preprocess = clip.load('ViT-B/32', device)

Using device: cuda


In [ ]:
import pickle

dataset = 'RESICS45'
dataset_dir = 'data_'
classes, seen_classes, unseen_classes = get_class_names(dataset, dataset_dir, 500)
seen_classes = classes
unseen_classes = classes
data_folder = f"{dataset_dir}/{dataset}"
labeled_data, _, _ = get_labeled_and_unlabeled_data(
    dataset, data_folder, seen_classes, unseen_classes, classes
)
labeled_files, labeles = zip(*labeled_data)
labeled_files = np.array(labeled_files)
labeles = np.array(labeles)
label_to_idx = {c: idx for idx, c in enumerate(classes)}
labeled_files = [os.path.join(data_folder, filepath) for filepath in labeled_files]

print(f"Number of labeled data: {len(labeled_files)}")
print(labeled_files[0])

filename = f"pseudolabels/{dataset}_ViT-B32_textual_fpl_pseudolabels_split_500.pickle"
with open(filename, 'rb') as f:
    data = pickle.load(f)
filepaths = data['filepaths']
probs = data['probs']
image_features = data['image_features']
text_features = data['text_features']
print(len(filepaths), len(probs), len(image_features), len(text_features))

# 将data中的列表按照labeled_files中的顺序重新排序
idx_map = {os.path.basename(filepath): i for i, filepath in enumerate(labeled_files)}
filepaths = [filepaths[idx_map[os.path.basename(filepath)]] for filepath in labeled_files]
probs = [probs[idx_map[os.path.basename(filepath)]] for filepath in labeled_files]
all_true_labels = labeles.tolist()
all_predicted_labels = [classes[prob.argmax()] for prob in probs]
image_features = [image_features[idx_map[os.path.basename(filepath)]] for filepath in labeled_files]
with torch.no_grad():
    all_logits = (model.logit_scale.exp() * torch.tensor(image_features).to(device) @ torch.tensor(text_features).to(device).t()).cpu().numpy()


In [9]:
def loop_clustering(classes, image_features, text_features):

    cluster_for_class = {}
    samples_for_class = {i: [] for i in range(len(classes))}
    remaining_image_features = [i for i in range(len(image_features))]
    remaining_classes = [i for i in range(len(classes))]

    logit_scale = model.logit_scale.exp().detach()
    text_features = torch.tensor(text_features).to(device)
    avg_size = int(len(image_features) / len(classes))

    while True:
        # Perform KMeans clustering
        kmeans = KMeans(n_clusters=len(remaining_classes), random_state=42)
        image_features_for_clustering = [image_features[i] for i in remaining_image_features]
        num_clusters = len(remaining_classes)
        cluster_labels = kmeans.fit_predict(image_features_for_clustering)
        cluster_centers = torch.tensor(kmeans.cluster_centers_, dtype=text_features.dtype).to(device)
        true_labels = [all_true_labels[i] for i in remaining_image_features]
        predicted_labels = [all_predicted_labels[i] for i in remaining_image_features]
        remaining_classes_names = [classes[i] for i in remaining_classes]
        # visualize_clustering_results(true_labels, predicted_labels, cluster_labels, classes, num_clusters=len(remaining_classes))
        # Normalize cluster centers
        cluster_centers /= torch.norm(cluster_centers, dim=-1, keepdim=True)

        text_features_for_remaining_classes = torch.stack([text_features[i] for i in remaining_classes]).to(device)
        image_features_for_remaining_classes = torch.tensor(image_features_for_clustering, dtype=text_features.dtype).to(device)
        # Compute cosine similarity between text features and normalized cluster centers
        similarity_for_centers = logit_scale * text_features_for_remaining_classes @ cluster_centers.t()
        probs_of_centers = similarity_for_centers.softmax(dim=-1).cpu()
        similarity_for_images = logit_scale * image_features_for_remaining_classes @ text_features_for_remaining_classes.t()
        probs_of_images = similarity_for_images.softmax(dim=-1).cpu()

        used_image_features = set()
        used_classes = set()
        center_confidences = probs_of_centers

        leaderboard = compute_leaderboard(probs_of_images, 100000, remaining_classes, remaining_image_features)

        confidences = center_confidences
        
        if len(used_classes) == 0:
            max_conf = confidences.max()
            for i in range(confidences.shape[0]):
                if confidences[i].max() == max_conf:
                    most_confident_class_idx = i
                    break
            cluster_for_class[remaining_classes[most_confident_class_idx]] = confidences[most_confident_class_idx].argmax().item()
            samples = leaderboard[remaining_classes[most_confident_class_idx]]
            samples.sort(reverse=True)
            for score, i in samples:
                if cluster_labels[i] == cluster_for_class[remaining_classes[most_confident_class_idx]] and len(samples_for_class[remaining_classes[most_confident_class_idx]]) < avg_size:
                    samples_for_class[remaining_classes[most_confident_class_idx]].append((score, remaining_image_features[i]))
                    used_image_features.add(remaining_image_features[i])
            used_classes.add(remaining_classes[most_confident_class_idx])
            for score, i in samples:
                if remaining_image_features[i] not in used_image_features and len(used_image_features) < avg_size:
                    # samples_for_class[remaining_classes[most_confident_class_idx]].append((score, remaining_image_features[i]))
                    used_image_features.add(remaining_image_features[i])

        remaining_image_features = [i for i in remaining_image_features if i not in used_image_features]
        remaining_classes = [i for i in remaining_classes if i not in used_classes]

        print(f"{[(classes[i], cluster_for_class[i]) for i in used_classes]}")
        print(f"remaining classes: {remaining_classes}")
        print(len(used_image_features))  

        # if len(remaining_classes) <= ceil(len(classes) / 10):  
        if len(remaining_classes) == 0:
            return remaining_classes, samples_for_class

In [ ]:
def compute_leaderboard(probs, k, remaining_classes, remaining_samples):
    # to find the top k for each class, each class has it's own "leaderboard"
    top_k_leaderboard = {
        label_to_idx[classes[i]]: [] for i in remaining_classes
    }
    for i in range(len(remaining_samples)):
        """if predicted class has empty leaderboard, or if the confidence is high
        enough for predicted class leaderboard, add the new example
        """
        prob = probs[i]
        pred = remaining_classes[prob.argmax().item()]
        prob_score = prob.max().item()
        if len(top_k_leaderboard[pred]) < k:
            top_k_leaderboard[pred].append((prob_score, i))
        elif (
            top_k_leaderboard[pred][-1][0] < prob_score
        ):
            top_k_leaderboard[pred] = sorted(
                top_k_leaderboard[pred] + [(prob_score, i)],
                reverse=True,
            )[:k]
        
    return top_k_leaderboard

In [ ]:
remaining_classes, samples_for_class = loop_clustering(classes, image_features, text_features)